# Feature Engineering

Feature Engineering = die Kunst, **rohe Daten** in eine Form zu bringen, die ML-Modelle verstehen und nutzen können.

**Garbage in, garbage out** — selbst das beste Modell liefert schlechte Ergebnisse mit schlechten Daten.

## Inhaltsverzeichnis
1. Warum Feature Engineering?
2. One-Hot Encoding (kategorische → numerische Variablen)
3. Label Encoding
4. Normalisierung (MinMaxScaler)
5. Standardisierung (StandardScaler)
6. Korrelationsanalyse & Feature-Selektion
7. Binning (Zahlen → Kategorien)
8. Polynomiale Features


## 1. Warum Feature Engineering?

ML-Algorithmen verstehen nur **Zahlen**. Aber echte Daten enthalten:
- Kategorien (z.B. "Berlin", "Wien", "Zürich")
- Text (z.B. Produktbeschreibungen)
- Verschiedene Maßstäbe (z.B. Alter: 0-100 vs. Gehalt: 0-200.000)
- Fehlende Werte

**Feature Engineering löst diese Probleme!**

Außerdem können wir durch cleveres Feature Engineering **neue Merkmale** erstellen, die das Modell besser machen — z.B. "Alter des Autos" aus Baujahr berechnen.


## 2. One-Hot Encoding

Kategorische Variablen ohne natürliche Reihenfolge müssen als **Binärspalten** kodiert werden.

**Warum nicht einfach Zahlen (0, 1, 2)?**  
Das würde eine künstliche Reihenfolge einführen — das Modell würde denken "Wien > Berlin"!

**Beispiel:**
```
Farbe: Rot, Blau, Grün
→ Farbe_Rot: 1/0, Farbe_Blau: 1/0, Farbe_Grün: 1/0
```

**Drop First:** Eine Spalte kann man weglassen (sie ist aus den anderen ableitbar) um Multikollinearität zu vermeiden.


In [ ]:
import pandas as pd
import numpy as np

# Beispieldaten
daten = pd.DataFrame({
    'Farbe': ['Rot', 'Blau', 'Grün', 'Rot', 'Blau'],
    'Größe': [1.70, 1.85, 1.65, 1.90, 1.75],
    'Preis': [100, 150, 80, 200, 120]
})

print("Originaldaten:")
print(daten)
print()

# One-Hot Encoding
encoded = pd.get_dummies(daten['Farbe'], prefix='Farbe', drop_first=True, dtype=int)
print("Nach One-Hot Encoding:")
print(encoded)

In [ ]:
# Komplette Transformation
daten_komplett = pd.concat([daten.drop('Farbe', axis=1), encoded], axis=1)
print("Fertige Daten (nur Zahlen):")
print(daten_komplett)

## 3. Label Encoding

Für **ordinale** Kategorien (mit natürlicher Reihenfolge) können wir einfach Zahlen vergeben.

**Beispiele für ordinale Kategorien:**
- Junior (0) < Senior (1) ← Reihenfolge macht Sinn
- Klein (0) < Mittel (1) < Groß (2)

**Achtung:** Nur verwenden wenn wirklich eine Reihenfolge existiert!


In [ ]:
# Label Encoding für ordinale Variable
erfahrung_map = {'Junior': 0, 'Senior': 1}

hr_daten = pd.DataFrame({
    'Name': ['Anna', 'Bob', 'Clara'],
    'Erfahrung': ['Junior', 'Senior', 'Junior'],
    'Gehalt': [45000, 85000, 48000]
})

hr_daten['Erfahrung_code'] = hr_daten['Erfahrung'].map(erfahrung_map)
print(hr_daten)

## 4. Normalisierung — MinMaxScaler

**Problem:** KNN und SVM sind abstandsbasiert. Wenn ein Merkmal Werte von 0-100.000 hat und ein anderes von 0-1, dann dominiert das erste Merkmal komplett!

**MinMax-Normalisierung:** Skaliert alle Werte auf den Bereich [0, 1]:

```
x_normalisiert = (x - x_min) / (x_max - x_min)
```

**Wichtige Regel:** Scaler nur auf **Trainingsdaten** fitten, dann auf Test- und Produktionsdaten anwenden!  
(Sonst "leakt" Information aus dem Testset ins Training)


In [ ]:
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

# Beispiel: Merkmale mit sehr unterschiedlichen Skalen
X_beispiel = pd.DataFrame({
    'Alter': [25, 45, 35, 55, 28],
    'Gehalt': [30000, 80000, 55000, 120000, 35000],
    'Erfahrung_jahre': [2, 20, 10, 30, 3]
})

print("Vor der Normalisierung:")
print(X_beispiel)
print()

scaler = MinMaxScaler()
scaler.fit(X_beispiel)  # Lernt min und max aus den Daten
X_normalisiert = scaler.transform(X_beispiel)
X_norm_df = pd.DataFrame(X_normalisiert, columns=X_beispiel.columns)

print("Nach der Normalisierung (alle Werte zwischen 0 und 1):")
print(X_norm_df.round(3))

## 5. Standardisierung — StandardScaler

**Alternative zur Normalisierung:**

**StandardScaler** transformiert die Daten so, dass sie:
- Mittelwert = 0 haben
- Standardabweichung = 1 haben

```
x_standard = (x - Mittelwert) / Standardabweichung
```

**Wann welchen?**
- **MinMaxScaler**: Wenn bekannter Wertebereich, keine Ausreißer
- **StandardScaler**: Wenn Ausreißer vorhanden, oder für Ridge/Lasso (MUSS bei Regularisierung!)


In [ ]:
from sklearn.preprocessing import StandardScaler

standard_scaler = StandardScaler()
standard_scaler.fit(X_beispiel)
X_standard = standard_scaler.transform(X_beispiel)
X_std_df = pd.DataFrame(X_standard, columns=X_beispiel.columns)

print("Nach StandardScaler (Mittelwert=0, Std=1):")
print(X_std_df.round(3))
print()
print(f"Mittelwerte: {X_std_df.mean().round(3).to_dict()}")
print(f"Standardabw: {X_std_df.std().round(3).to_dict()}")

## 6. Korrelationsanalyse & Feature-Selektion

**Warum Feature-Selektion?**
1. Modell wird einfacher und schneller
2. Weniger Overfitting
3. Bessere Interpretierbarkeit

**Zwei Strategien:**
- **Hochkorrelierte Features entfernen:** Wenn zwei Features fast dasselbe messen, braucht man nur eines
- **Wenig korrelierte Features mit Ziel entfernen:** Features die kaum Vorhersagekraft haben


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.datasets import load_boston
from sklearn.datasets import fetch_openml

dataset = fetch_openml(name='boston', version=1)
X_boston = dataset.data.astype(float)
y_boston = dataset.target.astype(float)

# Korrelationsmatrix
plt.figure(figsize=(12, 10))
korr_matrix = X_boston.corr()
mask = np.triu(np.ones_like(korr_matrix, dtype=bool))
sns.heatmap(korr_matrix, mask=mask, annot=True, fmt='.1f', cmap='coolwarm',
            center=0, vmin=-1, vmax=1, linewidths=0.5)
plt.title("Korrelationsmatrix der Features")
plt.tight_layout()
plt.show()

In [ ]:
# Features mit Ziel korreliert?
korr_mit_ziel = X_boston.apply(lambda col: col.corr(y_boston)).sort_values()

plt.figure(figsize=(8, 6))
korr_mit_ziel.plot(kind='barh', color=korr_mit_ziel.map(lambda x: 'green' if x > 0 else 'red'))
plt.axvline(x=0, color='black', linewidth=0.5)
plt.xlabel("Korrelation mit Hauspreis")
plt.title("Welche Features hängen mit dem Preis zusammen?")
plt.grid(True, alpha=0.3)
plt.show()

print("Stärkste positive Korrelation (steigender Preis):")
print(korr_mit_ziel.tail(3))
print("\nStärkste negative Korrelation (sinkender Preis):")
print(korr_mit_ziel.head(3))

## 7. Binning (Zahlen → Kategorien)

Manchmal ist es sinnvoll, eine Zahl in Kategorien aufzuteilen:
- Alter → "jung/mittel/alt"  
- Einkommen → "niedrig/mittel/hoch"

**Wann nützlich?**
- Wenn die Beziehung nicht-linear ist (z.B. Risiko bei jungen UND alten Fahrern)
- Für bessere Interpretierbarkeit


In [ ]:
# Beispiel: Alter in Gruppen einteilen
alter = pd.Series([18, 25, 35, 42, 55, 65, 72, 28, 45, 38])

alter_gruppen = pd.cut(alter, 
                       bins=[0, 25, 40, 60, 100],
                       labels=['Jugend (< 25)', 'Jung-Erwachsen (25-40)', 
                               'Mittelalt (40-60)', 'Senior (60+)'])

vergleich = pd.DataFrame({'Alter': alter, 'Gruppe': alter_gruppen})
print(vergleich)

## 8. Polynomiale Features

**Idee:** Aus bestehenden Features neue erstellen durch Potenzierung und Multiplikation.

**Beispiel mit 2 Features (a, b):**
- Grad 2 ergibt: a, b, a², ab, b²
- Grad 3 ergibt: a, b, a², ab, b², a³, a²b, ab², b³

**Warum?**  
Lineare Regression kann nur lineare Beziehungen modellieren. Mit polynomialen Features kann sie auch Kurven modellieren!

**Achtung:** Bei zu hohem Grad → Overfitting!


In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
import numpy as np

# Künstliche Daten mit nicht-linearem Zusammenhang
np.random.seed(42)
X_poly = np.linspace(-3, 3, 100).reshape(-1, 1)
y_poly = X_poly.flatten()**2 + np.random.normal(0, 0.5, 100)

X_tr, X_te, y_tr, y_te = train_test_split(X_poly, y_poly, test_size=0.2, random_state=42)

plt.figure(figsize=(14, 4))

for i, grad in enumerate([1, 2, 5]):
    poly = PolynomialFeatures(degree=grad)
    X_tr_poly = poly.fit_transform(X_tr)
    X_te_poly = poly.transform(X_te)
    
    modell = LinearRegression().fit(X_tr_poly, y_tr)
    
    X_plot = np.linspace(-3, 3, 300).reshape(-1, 1)
    X_plot_poly = poly.transform(X_plot)
    y_plot = modell.predict(X_plot_poly)
    
    ax = plt.subplot(1, 3, i+1)
    ax.scatter(X_tr, y_tr, alpha=0.5, s=15)
    ax.plot(X_plot, y_plot, 'r-', lw=2)
    ax.set_title(f"Grad {grad}\nTest R²: {r2_score(y_te, modell.predict(X_te_poly)):.3f}")
    ax.grid(True, alpha=0.3)

plt.suptitle("Polynomiale Features: Grad 1 = zu simpel, Grad 5 = Overfitting", y=1.05)
plt.tight_layout()
plt.show()

## Zusammenfassung: Feature Engineering Checkliste

Vor dem Modelltraining immer prüfen:

- [ ] **Kategorische Variablen enkodiert?** (One-Hot oder Label Encoding)
- [ ] **Skalierung nötig?** (bei KNN, SVM, Ridge/Lasso: Ja!)
- [ ] **Hoch-korrelierte Features entfernt?** (r > 0.9 → eine reicht)
- [ ] **Fehlende Werte behandelt?** (Imputation oder Entfernung)
- [ ] **Ausreißer geprüft?** (können Modelle stark beeinflussen)
- [ ] **Neue Features sinnvoll?** (z.B. Verhältnisse, Differenzen)

**Wichtigste Regel:** Alle Transformationen nur auf Trainingsdaten **lernen** (`.fit()`), dann auf alle Daten **anwenden** (`.transform()`)!
